# Phase 2 — Bronze Layer Ingestion
## Recidivism Prediction | Medallion Architecture

> ⚠️ **This notebook runs in Databricks Free Edition with Serverless compute.**
> Import this file into your Databricks Workspace and run cells top to bottom.
> Serverless compute starts automatically — no cluster needed.

**Goal:** Ingest the raw ProPublica COMPAS CSV into the Bronze layer of the Medallion architecture.

**Medallion layer:** 🟤 Bronze — raw data, exactly as received. No cleaning. No transformation.

**Key concepts:** Delta Lake, ACID transactions, Unity Catalog Volumes, PySpark DataFrames, time travel.

## 1. Verify Environment

In [ ]:
# Confirm Spark session (auto-created by Serverless compute — no cluster needed)
print(f"Spark version:  {spark.version}")
print(f"App name:       {spark.sparkContext.appName}")
print()
print("Serverless compute confirmed — Spark session is live.")

In [ ]:
import os

# Path to the CSV uploaded via Catalog → Add Data → Upload files to volume
CSV_PATH = "/Volumes/workspace/default/recidivai/compas-scores-two-years-violent.csv"

if os.path.exists(CSV_PATH):
    size_mb = os.path.getsize(CSV_PATH) / 1024 / 1024
    print(f"✓ Found: {CSV_PATH}")
    print(f"  Size:  {size_mb:.2f} MB")
else:
    print(f"⚠️  File not found at: {CSV_PATH}")

## 2. Read Raw CSV into a Spark DataFrame

`inferSchema=True` triggers a full file scan to detect column types.
Fine for 4,000 rows — in production you'd define an explicit `StructType` schema to skip the scan cost.

> **Interview concept — inferSchema trade-off:**
> Convenient but costs one full file scan. Production code uses explicit `StructType`
> to skip the scan and guarantee type safety. We demonstrate explicit schemas in Silver.

In [ ]:
df_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH))

print(f"Rows:    {df_raw.count():,}")
print(f"Columns: {len(df_raw.columns)}")
print()
df_raw.printSchema()

In [ ]:
df_raw.show(5, truncate=False)

In [ ]:
df_raw.describe().show(truncate=False)

## 3. Null & Sentinel Value Audit

Document data quality at Bronze — but do NOT fix anything here.
Bronze = raw data preserved exactly as received. Cleaning happens in Silver.

> **Why audit at Bronze?** It creates a baseline. After Silver cleaning, you can compare
> null counts between Bronze and Silver to verify each cleaning step had the intended effect.

In [ ]:
from pyspark.sql import functions as F

print("Null counts per column (Bronze — raw):")
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]
df_raw.select(null_exprs).show(truncate=False)

In [ ]:
# Sentinel value check: v_decile_score <= 0 means invalid COMPAS assessment
print("v_decile_score distribution (sentinel check):")
(df_raw
    .groupBy("v_decile_score")
    .count()
    .orderBy("v_decile_score")
    .show(15))

invalid = df_raw.filter(F.col("v_decile_score") <= 0).count()
print(f"Rows with invalid COMPAS score (≤ 0): {invalid}  — will be filtered in Silver.")

## 4. Define Medallion Layer Table Names

Unity Catalog uses a three-level namespace: `catalog.schema.table`.
All three Medallion layers live under `workspace.recidivism`.

In [ ]:
CATALOG = "workspace"
SCHEMA  = "recidivism"
BRONZE  = f"{CATALOG}.{SCHEMA}.bronze"
SILVER  = f"{CATALOG}.{SCHEMA}.silver"   # Phase 3
GOLD    = f"{CATALOG}.{SCHEMA}.gold"     # Phase 4

print(f"Bronze: {BRONZE}")
print(f"Silver: {SILVER}")
print(f"Gold:   {GOLD}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"\n✓ Schema '{CATALOG}.{SCHEMA}' ready.")

## 5. Write Raw Data to Bronze Delta Table

Core Bronze step: write the raw DataFrame exactly as ingested — no transformations.

> **Interview concept — Why Delta over plain Parquet?**
> Parquet is just files in a folder with no coordination layer. Delta adds:
> - **`_delta_log/`** — a JSON transaction log recording every write
> - **ACID guarantees** — failed writes don't corrupt the table
> - **Schema enforcement** — future writes that don't match are rejected
> - **Time travel** — `VERSION AS OF 0` always returns this exact raw snapshot

In [ ]:
print(f"Writing {df_raw.count():,} rows to: {BRONZE}")

(df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE))

print(f"\n✓ Bronze table written successfully: {BRONZE}")

## 6. Verify the Write

In [ ]:
df_bronze = spark.table(BRONZE)

source_count = df_raw.count()
bronze_count = df_bronze.count()
status = "✓ MATCH" if source_count == bronze_count else "✗ MISMATCH"

print(f"Source CSV rows:    {source_count:,}")
print(f"Bronze table rows:  {bronze_count:,}")
print(f"Integrity check:    {status}")
print()
df_bronze.show(5)

In [ ]:
# Inspect the Delta transaction log — confirms ACID write + time travel capability
spark.sql(f"DESCRIBE HISTORY {BRONZE}").show(truncate=False)

## 7. Query Bronze with SQL

This is the ELT pattern: load raw first (Bronze), transform later using the platform's query engine.
The table is now queryable from any Databricks notebook with plain SQL.

> **Interview concept — ETL vs ELT:**
> ETL transforms before loading (traditional warehouses). ELT loads raw first, transforms
> inside the platform (Databricks, Snowflake, BigQuery). We use ELT because object storage
> is cheap and Spark compute is elastic — preserving raw data lets you reprocess if rules change.

In [ ]:
spark.sql(f"""
    SELECT
        race,
        COUNT(*)                                AS n,
        SUM(is_violent_recid)                   AS n_reoffended,
        ROUND(AVG(is_violent_recid) * 100, 2)   AS reoffense_rate_pct
    FROM {BRONZE}
    GROUP BY race
    ORDER BY reoffense_rate_pct DESC
""").show()

## 8. Phase 2 Summary

### Accomplished
- ✓ CSV read from Unity Catalog Volume into Spark DataFrame
- ✓ Null and sentinel value audit documented at Bronze layer
- ✓ Raw data written to Delta table: `workspace.recidivism.bronze`
- ✓ Row count integrity verified (source CSV = Bronze table)
- ✓ Delta transaction log inspected — ACID write confirmed, time travel available
- ✓ SQL query confirmed — table queryable via Unity Catalog

### Key interview points
- **Bronze = raw, immutable.** No cleaning here — that's Silver's job.
- **Delta vs Parquet:** Delta adds `_delta_log/`, ACID, schema enforcement, time travel.
- **Unity Catalog:** Three-level namespace (`catalog.schema.table`).
- **Serverless compute:** No cluster management — Databricks provisions Spark automatically.
- **ELT confirmed:** Load raw first (Bronze), transform later (Silver/Gold).

### Next step
**Phase 3:** Silver layer — PySpark ELT pipeline. Clean, cast, deduplicate, write `workspace.recidivism.silver`.

---
*Notebook: Phase 2 — Bronze Ingestion | Project: Recidivism Prediction | Arjun, UC Berkeley*